# 04 — SQL Export to PostgreSQLThis notebook takes the feature-engineered DataFrame and loads it into a PostgreSQLdatabase table called `jobs`. This is the table that `analysis_queries.sql` and thePower BI dashboard both read from.**Requires:** PostgreSQL running locally with a database called `uk_jobs_db` (see `schema.sql`),and a `.env` file with `DB_USER`, `DB_PASSWORD`, `DB_HOST`, `DB_PORT`, `DB_NAME`.

In [1]:
import pandas as pdimport sqlalchemyfrom pathlib import Pathimport osfrom dotenv import load_dotenvBASE_DIR = Path.cwd()load_dotenv(BASE_DIR / ".env")df = pd.read_csv(BASE_DIR / "data" / "engineered" / "uk_jobs_features.csv")print("Rows ready to load:", len(df))

Rows ready to load: 726

In [2]:
DB_USER     = os.getenv("DB_USER", "postgres")DB_PASSWORD = os.getenv("DB_PASSWORD")DB_HOST     = os.getenv("DB_HOST", "localhost")DB_PORT     = os.getenv("DB_PORT", "5432")DB_NAME     = os.getenv("DB_NAME", "uk_jobs_db")connection_string = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"engine = sqlalchemy.create_engine(connection_string)print("Engine created for database:", DB_NAME)

Engine created for database: uk_jobs_db

In [3]:
jobs_table = df.rename(columns={    "location_clean": "company_location"})[[    "job_id", "job_title", "company", "company_location", "country",    "employment_type", "work_type", "is_remote", "date_posted",    "salary_usd", "experience_category", "role_category",    "company_size_clean", "benefits_score", "job_description"]]jobs_table.head()

(726 rows x 15 columns)

In [4]:
jobs_table.to_sql(    name="jobs",    con=engine,    if_exists="replace",    index=False)print("Data successfully written to PostgreSQL!")print(f"Total rows loaded into 'jobs' table: {len(jobs_table)}")

Data successfully written to PostgreSQL!\nTotal rows loaded into 'jobs' table: 726

In [5]:
check = pd.read_sql("SELECT COUNT(*) AS total_jobs FROM jobs;", engine)check

   total_jobs\n0         726

**Output of this notebook:** A PostgreSQL table called `jobs` containing the cleaned, feature-engineered job postings. This table powers `analysis_queries.sql` and the Power BI **Executive Overview**, **Skills Intelligence**, and **Skills & Salary Intelligence** pages.